# Griffin & Hawk — a toy-scale build of a real-gated recurrence + local attention hybrid

A minimal implementation of the **RG-LRU** (Real-Gated Linear Recurrent
Unit) and the **Griffin** architecture, from De, Smith, Fernando, et al.
(Google DeepMind), *"Griffin: Mixing Gated Linear Recurrences with Local
Attention for Efficient Language Models"* (2024).

This is the first **hybrid** architecture in this repo: every other
recurrent-state notebook (KDA, GLA, RetNet, DeltaNet, Mamba, xLSTM) replaces
attention entirely. Griffin doesn't — it interleaves a gated recurrence with
occasional *local* (sliding-window) attention layers, on the view that
recurrence is good at compressing long-range history and attention is good
at precise short-range recall, and a model might as well get both.

Companion write-up: `README.md` in this folder.

## 0. Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

## 1. The RG-LRU: a *vector*-state recurrence

Every other gated recurrent layer in this repo (KDA, GLA, RetNet, DeltaNet,
xLSTM) keeps a **matrix** state — built from an outer product of a key and
a value, so it can associate specific keys with specific values, the way
attention does. The RG-LRU takes a different, simpler approach: its state
is just a **vector**, one number per channel, with no keys or values
involved at all. It's much closer to a classic gated exponential moving
average than to attention.

At every timestep, two gates are computed from the input:

- `r_t` (the **recurrence gate**) — controls how much this channel's decay
  rate should shift, per token.
- `i_t` (the **input gate**) — controls how much of the new input actually
  gets let in, independent of the recurrence gate.

These combine with a fixed, learned per-channel base decay `a` (in `(0,1)`)
to produce a per-token, per-channel effective decay:

```
a_t = a ^ (c * r_t)        # c is a constant (8 in the paper); r_t in (0,1) modulates how close a_t gets to 1
h_t = a_t * h_{t-1} + sqrt(1 - a_t^2) * (i_t * x_t)
```

The `sqrt(1 - a_t^2)` term is a **variance-preserving correction** — without
it, a channel with a decay close to 0 (fast forgetting) would inject far
more variance per step than a channel with decay close to 1 (slow
forgetting), which makes training less stable across channels with very
different effective memory lengths. This term rescales the new input's
contribution so that, on average, the state's variance stays roughly
constant regardless of the decay rate a given channel happens to be using.

A full **Hawk block** (Griffin's purely-recurrent building block) wraps this
recurrence the same way Mamba wraps its SSM: project up into two branches,
run a short causal convolution + the RG-LRU on one branch, gate with a
SiLU-activated version of the other branch, project back down.

## 2. Griffin: interleaving RG-LRU with local attention

**Hawk** is the model you get from stacking Hawk blocks alone — no
attention anywhere. **Griffin** takes the same Hawk blocks and periodically
swaps some of them out for a **local (sliding-window) attention** block —
ordinary causal softmax attention, just capped to only look back a fixed
number of tokens. The paper uses a repeating pattern of mostly-recurrent
blocks with attention blocks mixed in periodically (this notebook uses 2
Hawk blocks for every 1 local-attention block).

Why bother with attention at all, if the RG-LRU already handles long-range
information? The two mechanisms are good at different things:

- **RG-LRU** compresses the *entire* history into a fixed-size vector state
  — cheap and unbounded in context length, but lossy, since everything has
  to be squeezed into the same fixed number of channels.
- **Local attention** gives *exact*, lossless recall of the last `window`
  tokens — nothing gets compressed or forgotten within that window, but it
  can't see anything further back.

Mixing the two gives a model that's cheap like a recurrent model over long
context, while still getting precise short-range recall where it matters
most (nearby tokens are usually the most syntactically and locally
relevant ones anyway).

> **Simplification used here:** the paper also introduces custom,
> hardware-aware kernels for the RG-LRU recurrence (a real-valued analogue
> of the complex-valued linear recurrent unit from the earlier "LRU" paper,
> designed to run efficiently associatively-scanned on a TPU/GPU). This
> notebook uses the plain sequential recurrence — a Python loop over
> timesteps — for the RG-LRU, same as every other recurrent layer in this
> repo.

In [ ]:
class RGLRU(nn.Module):
    def __init__(self, d_model, c=8.0):
        super().__init__()
        self.c = c
        self.input_gate_proj = nn.Linear(d_model, d_model, bias=True)       # i_t
        self.recurrence_gate_proj = nn.Linear(d_model, d_model, bias=True)  # r_t
        self.log_lambda = nn.Parameter(torch.linspace(-4.0, -0.1, d_model))  # per-channel base decay (pre-sigmoid)

    def forward(self, x):
        B, T, D = x.shape
        r = torch.sigmoid(self.recurrence_gate_proj(x))
        i_gate = torch.sigmoid(self.input_gate_proj(x))
        a_base = torch.sigmoid(self.log_lambda).view(1, 1, D)
        a = a_base ** (self.c * r)                                    # effective per-token, per-channel decay

        h = x.new_zeros(B, D)
        outs = []
        for t in range(T):
            a_t, i_t, x_t = a[:, t], i_gate[:, t], x[:, t]
            h = a_t * h + torch.sqrt((1 - a_t ** 2).clamp_min(1e-6)) * (i_t * x_t)   # variance-preserving update
            outs.append(h)
        return torch.stack(outs, dim=1)

class HawkBlock(nn.Module):
    '''Purely-recurrent block: project up, short conv, RG-LRU, gate, project down.'''
    def __init__(self, d_model=64, d_inner=128, conv_kernel=4):
        super().__init__()
        self.in_proj = nn.Linear(d_model, 2 * d_inner, bias=False)
        self.conv = nn.Conv1d(d_inner, d_inner, conv_kernel, groups=d_inner, padding=0)
        self.conv_kernel = conv_kernel
        self.rglru = RGLRU(d_inner)
        self.out_proj = nn.Linear(d_inner, d_model, bias=False)

    def forward(self, x):
        xz = self.in_proj(x)
        x_b, z_b = xz.chunk(2, dim=-1)
        xc = x_b.transpose(1, 2)
        xc = F.pad(xc, (self.conv_kernel - 1, 0))
        x_b = F.silu(self.conv(xc).transpose(1, 2))
        y = self.rglru(x_b)
        y = y * F.silu(z_b)
        return self.out_proj(y)

In [ ]:
class LocalAttention(nn.Module):
    '''Ordinary causal attention, capped to a sliding window -- Griffin's non-recurrent block.'''
    def __init__(self, d_model=64, n_heads=2, d_head=32, window=8):
        super().__init__()
        self.h, self.dh, self.win = n_heads, d_head, window
        inner = n_heads * d_head
        self.q_proj = nn.Linear(d_model, inner, bias=False)
        self.k_proj = nn.Linear(d_model, inner, bias=False)
        self.v_proj = nn.Linear(d_model, inner, bias=False)
        self.out_proj = nn.Linear(inner, d_model, bias=False)
        self.scale = d_head ** -0.5

    def forward(self, x):
        B, T, D = x.shape
        H, Dh = self.h, self.dh
        q = self.q_proj(x).view(B, T, H, Dh).transpose(1, 2)
        k = self.k_proj(x).view(B, T, H, Dh).transpose(1, 2)
        v = self.v_proj(x).view(B, T, H, Dh).transpose(1, 2)
        scores = torch.einsum('bhtd,bhsd->bhts', q, k) * self.scale
        pos_diff = torch.arange(T, device=x.device).view(-1, 1) - torch.arange(T, device=x.device).view(1, -1)
        mask = (pos_diff < 0) | (pos_diff >= self.win)     # causal AND within the local window
        attn = scores.masked_fill(mask, float('-inf')).softmax(-1)
        o = torch.einsum('bhts,bhsd->bhtd', attn, v).transpose(1, 2).reshape(B, T, H * Dh)
        return self.out_proj(o)

## 3. Assembling a tiny language model

`TinyLM` builds a repeating pattern of **2 Hawk blocks, then 1 local
attention block** — Griffin's actual recipe (the paper uses this same 2:1
ratio). Setting `griffin=False` below would build pure Hawk (recurrent-only)
instead, for comparison.

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        norm = x.pow(2).mean(-1, keepdim=True)
        return x * torch.rsqrt(norm + self.eps) * self.weight

class SwiGLU(nn.Module):
    def __init__(self, d, hidden_mult=2):
        super().__init__()
        h = d * hidden_mult
        self.Wg = nn.Linear(d, h, bias=False)
        self.Wu = nn.Linear(d, h, bias=False)
        self.Wd = nn.Linear(h, d, bias=False)
    def forward(self, x):
        return self.Wd(F.silu(self.Wg(x)) * self.Wu(x))

class TinyLM(nn.Module):
    def __init__(self, vocab_size, d_model=64, n_layers=3, griffin=True):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        # Griffin's recipe: 2 recurrent blocks for every 1 local-attention block.
        # With griffin=False, every layer is a HawkBlock (pure recurrence, Hawk's recipe).
        layer_types = []
        for i in range(n_layers):
            if griffin and (i % 3 == 2):
                layer_types.append('attn')
            else:
                layer_types.append('hawk')
        self.blocks = nn.ModuleList([
            LocalAttention(d_model) if t == 'attn' else HawkBlock(d_model) for t in layer_types
        ])
        self.mlps = nn.ModuleList([SwiGLU(d_model) for _ in range(n_layers)])
        self.norms1 = nn.ModuleList([RMSNorm(d_model) for _ in range(n_layers)])
        self.norms2 = nn.ModuleList([RMSNorm(d_model) for _ in range(n_layers)])
        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        print(f"Layer pattern: {layer_types}")

    def forward(self, idx):
        x = self.embed(idx)
        for blk, mlp, n1, n2 in zip(self.blocks, self.mlps, self.norms1, self.norms2):
            x = x + blk(n1(x))
            x = x + mlp(n2(x))
        return self.lm_head(self.final_norm(x))

## Proving it actually works

Everything above is only worth something if gradients actually flow correctly
through Griffin once it's wired into a real model. So the rest of this
notebook:

1. wraps Griffin into a tiny 2-layer causal language model,
2. builds a **tiny synthetic dataset** (a repeating `"0123456789ABCDEF"`
   string — enough to check the model can learn *any* sequential structure
   at all, no real corpus needed),
3. runs **one forward + backward pass** as a sanity check (right output
   shape, no `NaN` gradients),
4. **trains for a few hundred steps**, and
5. **generates** from the trained model — if training worked, the output
   should show visible periodicity.

This is deliberately not a "real" training run. It exists purely to catch
architecture bugs, which is the whole point of a toy-scale build.

In [ ]:
# --- synthetic dataset ---
pattern = "0123456789ABCDEF"      # synthetic, no copyright concerns
text = pattern * 200
chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
vocab_size = len(chars)
max_seq_len = 32

model = TinyLM(vocab_size).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model built. Trainable parameters: {n_params:,}")

In [ ]:
# --- sanity check: one forward + backward pass before training ---
xb0 = data[:max_seq_len].unsqueeze(0).to(device)
yb0 = data[1:max_seq_len + 1].unsqueeze(0).to(device)
out0 = model(xb0)
logits0 = out0[0] if isinstance(out0, tuple) else out0
print(f"Sanity check -- logits shape: {tuple(logits0.shape)} (expect [1, {max_seq_len}, {vocab_size}])")
loss0 = F.cross_entropy(logits0.reshape(-1, vocab_size), yb0.reshape(-1))
if isinstance(out0, tuple):
    loss0 = loss0 + out0[1]
loss0.backward()
n_nan_grads = sum(torch.isnan(p.grad).any().item() for p in model.parameters() if p.grad is not None)
print(f"Sanity check -- initial loss: {loss0.item():.4f}, NaN grads: {n_nan_grads}")
model.zero_grad()

In [ ]:
# --- training loop ---
def get_batch(data, block_size, batch_size, device):
    ix = torch.randint(0, len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
n_steps, batch_size = 300, 16
print("Training on synthetic periodic sequence (verifies grads flow end-to-end)...")
for step in range(n_steps):
    xb, yb = get_batch(data, max_seq_len, batch_size, device)
    out = model(xb)
    logits = out[0] if isinstance(out, tuple) else out
    loss = F.cross_entropy(logits.reshape(-1, vocab_size), yb.reshape(-1))
    if isinstance(out, tuple):
        loss = loss + out[1]
    opt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    if step % 50 == 0 or step == n_steps - 1:
        print(f"  step {step:4d} | loss {loss.item():.4f}")

In [ ]:
# --- generation ---
@torch.no_grad()
def generate(model, start_idx, n_new):
    model.eval()
    idx = start_idx.clone()
    for _ in range(n_new):
        out = model(idx)
        logits = out[0] if isinstance(out, tuple) else out
        probs = F.softmax(logits[:, -1, :], dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_id], dim=1)
    model.train()
    return idx

start = data[:8].unsqueeze(0).to(device)
gen = generate(model, start, 48)[0].tolist()
print("Generated (should show visible periodicity if training worked):")
print(''.join(itos[i] for i in gen))

## Where to go from here

- **Set `griffin=False`** in `TinyLM` to get pure Hawk (recurrent-only, no
  attention anywhere) and compare training against the mixed version — on
  this short, repetitive toy task the difference may be small, but it's a
  good place to start noticing where local attention actually earns its
  keep.
- **Compare the RG-LRU's vector state against KDA's matrix state.** The
  RG-LRU has no keys or values at all — just a gated per-channel running
  average. Try to reason about what kinds of sequences would favor one
  approach over the other (hint: think about tasks that need to associate
  specific *pairs* of things, versus tasks that just need a compressed
  running summary).
- **Widen or narrow the local attention window** and see how it trades off
  against the recurrent blocks' compressed long-range memory.

Reference: De, Smith, Fernando, Botev, Cristian-Muraru, Gu, Haroun, Berrada,
Chen, Srinivasan, Desjardins, Doucet, Budden, Teh, Pascanu, De Freitas,
Gulcehre, *"Griffin: Mixing Gated Linear Recurrences with Local Attention
for Efficient Language Models,"* 2024.